In [ ]:
import pandas as pd
import pickle
import os
import numpy as np

# Set base path for your macOS environment
base_path = "/Users/kevintao/Develop/DATA-481/ADCT_Article_Resource/predictability_performance_and_ISC/data"
merge_keys = ['teamID', 'sessionID', 'trialID', 'ringID']

def load_pkl(filename):
    with open(os.path.join(base_path, filename), 'rb') as f:
        return pickle.load(f)

# Load dataframes
df_eeg = load_pkl('epoched_eeg.pkl')
df_loc = load_pkl('epoched_location.pkl')
df_act = load_pkl('epoched_action.pkl')
df_perf = load_pkl('team_performance.pkl')

# Merge sequentially on ID keys only 
df_merged = df_eeg.merge(df_loc, on=merge_keys, how='inner')
df_merged = df_merged.merge(df_act, on=merge_keys, how='inner')
df_merged = df_merged.merge(df_perf, on=merge_keys, how='inner')

# Create performance scores: Easy=1, Medium=2, Hard=3
diff_map = {'Easy': 1, 'Medium': 2, 'Hard': 3}
# Use 'difficulty_x' or 'difficulty' depending on suffixes from your merge
diff_col = [c for c in df_merged.columns if 'difficulty' in c][0]
df_merged['target_score'] = df_merged[diff_col].map(diff_map)

# Keep only successful ring crossings
df_merged = df_merged.dropna(subset=['target_score'])
print(f"Merge Complete. Total successful rings: {len(df_merged)}")

Merge Complete. Total successful rings: 8822


In [ ]:
from sklearn.preprocessing import StandardScaler

def to_3d_array(series):
    return np.stack(series.tolist())

# Prepare EEG Branch (60 channels total) 
eeg_raw = np.stack([
    to_3d_array(df_merged['yawEEG']),
    to_3d_array(df_merged['pitchEEG']),
    to_3d_array(df_merged['thrustEEG'])
], axis=1).reshape(len(df_merged), 60, -1) 

# Prepare Action Branch (3 pilots)
act_raw = np.stack([
    to_3d_array(df_merged['yawAction']),
    to_3d_array(df_merged['pitchAction']),
    to_3d_array(df_merged['thrustAction'])
], axis=1).reshape(len(df_merged), 3, -1)

# Store time steps for model initialization
eeg_steps = eeg_raw.shape[2]
act_steps = act_raw.shape[2]

# Z-score Normalize EEG 
def normalize_eeg(data):
    s, c, t = data.shape
    scaler = StandardScaler()
    reshaped = data.transpose(0, 2, 1).reshape(-1, c)
    return scaler.fit_transform(reshaped).reshape(s, t, c).transpose(0, 2, 1)

eeg_norm = normalize_eeg(eeg_raw)
targets = df_merged['target_score'].values


import gc
del df_eeg, df_loc, df_act
gc.collect()

0

In [3]:
import torch
import torch.nn as nn

class TeamPerformanceCNN(nn.Module):
    def __init__(self, eeg_steps, act_steps):
        super(TeamPerformanceCNN, self).__init__()
        
        # EEG Branch: (60 channels, 384 time steps)
        self.eeg_branch = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, 3), padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        
        # Action Branch: (3 pilots, 90 time steps)
        self.act_branch = nn.Sequential(
            nn.Conv1d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten()
        )

        # Dynamic size calculation for Linear layer
        eeg_out = 16 * (60 // 2) * (eeg_steps // 2)
        act_out = 8 * (act_steps // 2)
        
        self.fc = nn.Sequential(
            nn.Linear(eeg_out + act_out, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1) # Regressing to Score 1, 2, or 3
        )

    def forward(self, eeg, act):
        x1 = self.eeg_branch(eeg.unsqueeze(1))
        x2 = self.act_branch(act)
        combined = torch.cat((x1, x2), dim=1)
        return self.fc(combined)

model = TeamPerformanceCNN(eeg_steps, act_steps)

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

# 1. Split data into Train and Validation (80/20)
full_ds = TensorDataset(torch.FloatTensor(eeg_norm), 
                        torch.FloatTensor(act_raw), 
                        torch.FloatTensor(targets))
train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

# 2. Setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 3. Training Loop with Tracking
for epoch in range(10):
    model.train()
    train_loss = 0
    for b_eeg, b_act, b_y in train_loader:
        b_eeg, b_act, b_y = b_eeg.to(device), b_act.to(device), b_y.to(device)
        
        optimizer.zero_grad()
        preds = model(b_eeg, b_act).squeeze()
        loss = criterion(preds, b_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # VALIDATION PHASE
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for b_eeg, b_act, b_y in val_loader:
            b_eeg, b_act, b_y = b_eeg.to(device), b_act.to(device), b_y.to(device)
            val_preds = model(b_eeg, b_act).squeeze()
            val_loss += criterion(val_preds, b_y).item()

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")